### Import des données

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(123)

In [4]:
%pip install s3fs
import s3fs

fs = s3fs.S3FileSystem(client_kwargs={"endpoint_url": "https://minio.lab.sspcloud.fr"})

MY_BUCKET = "mariedominge"
print(fs.ls(MY_BUCKET))


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
[]


In [38]:
with fs.open(f"{MY_BUCKET}/donnees_GBD_uniques.csv", "rb") as f:
    df_GBD = pd.read_csv(f, sep=";", encoding="latin-1", decimal=",")

df_GBD.head()

,population_group,measure,location_id,location_name,sex,age_name,cause_id,cause_name,metric_id,metric_name,year,value,upper_bound,lower_bound
0,All Population,Deaths,71,Australia,Both,All ages,294,All causes,1,Number,1980,108265.029600,108882.768100,107497.458600
1,All Population,Deaths,71,Australia,Both,All ages,491,Cardiovascular diseases,1,Number,1980,53252.643510,54951.178210,50203.750100
2,All Population,Deaths,71,Australia,Both,All ages,521,Cirrhosis and other chronic liver diseases,1,Number,1980,1390.286718,1469.238444,1314.555387
3,All Population,Deaths,71,Australia,Both,All ages,945,Conflict and terrorism,1,Number,1980,3.999771,4.397038,3.641975
4,All Population,Deaths,71,Australia,Both,All ages,724,Interpersonal violence,1,Number,1980,297.285701,330.648576,270.154420


### Modification du dataframe GBD pour faire correspondre les causes_GBD avec les causes_2006 (on passe de 14 causes à 9 causes)

In [51]:
# création d'un dataframe qui ne comporte que les infos utiles == qui different d'un pays à l'autre, etc 

df_GBD_modifie = df_GBD[["location_id", "location_name", "cause_id", "cause_name", "metric_id", "metric_name", "year", "value","upper_bound", "lower_bound"]]

# conversion du texte en valeurs pour value, upper_bound, lower_bound
df_GBD_modifie["value"] = pd.to_numeric(df_GBD_modifie["value"])
df_GBD_modifie["upper_bound"] = pd.to_numeric(df_GBD_modifie["upper_bound"])
df_GBD_modifie["lower_bound"] = pd.to_numeric(df_GBD_modifie["lower_bound"])

# ajout des écarts-types et variances pour après calculer les nouvelles upper_bound et lower_bound

df_GBD_modifie["ecart_type"] = (df_GBD_modifie["upper_bound"] - df_GBD_modifie["lower_bound"]) / (2 * 1.96)
df_GBD_modifie["var"] = df_GBD_modifie["ecart_type"]**2

df_GBD_modifie.head()

,location_id,location_name,cause_id,cause_name,metric_id,metric_name,year,value,upper_bound,lower_bound,ecart_type,var
0,71,Australia,294,All causes,1,Number,1980,108265.029600,108882.768100,107497.458600,353.395281,1.248882e+05
1,71,Australia,491,Cardiovascular diseases,1,Number,1980,53252.643510,54951.178210,50203.750100,1211.078599,1.466711e+06
2,71,Australia,521,Cirrhosis and other chronic liver diseases,1,Number,1980,1390.286718,1469.238444,1314.555387,39.459964,1.557089e+03
3,71,Australia,945,Conflict and terrorism,1,Number,1980,3.999771,4.397038,3.641975,0.192618,3.710169e-02
4,71,Australia,724,Interpersonal violence,1,Number,1980,297.285701,330.648576,270.154420,15.432183,2.381523e+02


#### Aggrégation pour Malignant neoplasm (2)

In [90]:
# filtration des causes qui relèvent de Malignant neoplasm 

cause_2 = [1012, 459, 489]

# pour chaque métrique, filtration & aggrégation & rajout du nombre de la métrique  

df_filtre_1 = df_GBD_modifie[(df_GBD_modifie["cause_id"].isin(cause_2)) & (df_GBD_modifie["metric_id"] == 1)]
df_aggreg_1 = df_filtre_1.groupby(["location_id","location_name", "year"], as_index=False)["value"].sum() 
df_aggreg_1 = df_aggreg_1.assign(
    metric_id = 1,
    metric_name = "Number"
)

df_filtre_2 = df_GBD_modifie[(df_GBD_modifie["cause_id"].isin(cause_2)) & (df_GBD_modifie["metric_id"] == 2)]
df_aggreg_2 = df_filtre_2.groupby(["location_id","location_name", "year"], as_index=False)["value"].sum() 
df_aggreg_2 = df_aggreg_2.assign(
    metric_id = 2,
    metric_name = "Percent"
)

df_filtre_3 = df_GBD_modifie[(df_GBD_modifie["cause_id"].isin(cause_2)) & (df_GBD_modifie["metric_id"] == 3)]
df_aggreg_3 = df_filtre_3.groupby(["location_id","location_name", "year"], as_index=False)["value"].sum() 
df_aggreg_3 = df_aggreg_3.assign(
    metric_id = 3,
    metric_name = "Rate"
)

df_aggreg_3.head()

,location_id,location_name,year,value,metric_id,metric_name
0,47,Czechia,1980,9.186061,3,Rate
1,47,Czechia,1981,9.108751,3,Rate
2,47,Czechia,1982,9.066288,3,Rate
3,47,Czechia,1983,9.237296,3,Rate
4,47,Czechia,1984,9.586514,3,Rate


In [ ]:
# Ajout des upper et lower bound pour chaque métrique (avec la métrique des variances)

df_aggreg_var_1 = df_filtre_1.groupby(["location_id","location_name", "year"], as_index=False)["var"].sum() 
df_aggreg_var_1["ecart_type"] = np.sqrt(df_aggreg_var_1["var"])
df_aggreg_var_2 = df_filtre_2.groupby(["location_id","location_name", "year"], as_index=False)["var"].sum() 
df_aggreg_var_2["ecart_type"] = np.sqrt(df_aggreg_var_2["var"])
df_aggreg_var_3 = df_filtre_3.groupby(["location_id","location_name", "year"], as_index=False)["var"].sum() 
df_aggreg_var_3["ecart_type"] = np.sqrt(df_aggreg_var_3["var"])

L1 = df_aggreg_var_1["ecart_type"].tolist()
L1 = [1.96 * x for x in L1]
L2 = df_aggreg_var_2["ecart_type"].tolist()
L2 = [1.96 * x for x in L2]
L3 = df_aggreg_var_3["ecart_type"].tolist()
L3 = [1.96 * x for x in L3]

df_aggreg_1["lower_bound"] = df_aggreg_1["value"] - L1
df_aggreg_1["upper_bound"] = df_aggreg_1["value"] + L1
df_aggreg_2["lower_bound"] = df_aggreg_2["value"] - L2
df_aggreg_2["upper_bound"] = df_aggreg_2["value"] + L2
df_aggreg_3["lower_bound"] = df_aggreg_3["value"] - L3
df_aggreg_3["upper_bound"] = df_aggreg_3["value"] + L3

df_aggreg_3.head()

,location_id,location_name,year,value,metric_id,metric_name,lower_bound,upper_bound
0,47,Czechia,1980,9.186061,3,Rate,8.550390,9.821731
1,47,Czechia,1981,9.108751,3,Rate,8.502427,9.715074
2,47,Czechia,1982,9.066288,3,Rate,8.483339,9.649238
3,47,Czechia,1983,9.237296,3,Rate,8.660633,9.813960
4,47,Czechia,1984,9.586514,3,Rate,9.023587,10.149441


In [ ]:
# concaténation des trois dataframe

df_aggreg = pd.concat([df_aggreg_1, df_aggreg_2, df_aggreg_3], ignore_index=True)

df_aggreg.head()

,location_id,location_name,year,value,metric_id,metric_name,lower_bound,upper_bound
0,47,Czechia,1980,943.567925,1,Number,878.273547,1008.862303
1,47,Czechia,1981,936.655643,1,Number,874.307188,999.004099
2,47,Czechia,1982,932.900040,1,Number,872.915908,992.884171
3,47,Czechia,1983,950.685882,1,Number,891.336740,1010.035024
4,47,Czechia,1984,986.545496,1,Number,928.614820,1044.476172
...,...,...,...,...,...,...,...,...
5011,155,Türkiye,2019,2.830499,3,Rate,2.342761,3.318236
5012,155,Türkiye,2020,2.923271,3,Rate,2.407299,3.439243
5013,155,Türkiye,2021,2.970430,3,Rate,2.419421,3.521439
5014,155,Türkiye,2022,3.161625,3,Rate,2.584937,3.738313


In [98]:
# rajout des autres colonnes et rangement 

df_aggreg = df_aggreg.assign(
    cause_id = 2,
    cause_name = "Malignant neoplasm",
)
df_aggreg = df_aggreg.sort_values(by=["location_name", "year","metric_id"]).reset_index(drop=True)


df_aggreg.head()

,location_id,location_name,year,value,metric_id,metric_name,lower_bound,upper_bound,cause_id,cause_name
0,71,Australia,1980,1109.779639,1,Number,1059.912533,1159.646745,2,Malignant neoplasm
1,71,Australia,1980,0.010251,2,Percent,0.009791,0.010710,2,Malignant neoplasm
2,71,Australia,1980,7.579365,3,Rate,7.238792,7.919938,2,Malignant neoplasm
3,71,Australia,1981,1229.501517,1,Number,1176.897735,1282.105300,2,Malignant neoplasm
4,71,Australia,1981,0.011188,2,Percent,0.010703,0.011673,2,Malignant neoplasm
